In [ ]:
import warnings
import numpy as np
import pandas as pd

from scipy.stats import mannwhitneyu

from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import RFECV
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import StratifiedKFold, GridSearchCV
from sklearn.metrics import (
    roc_auc_score,
    accuracy_score,
    confusion_matrix,
    f1_score,
    recall_score,
    precision_score,
    roc_curve
)

warnings.filterwarnings("ignore")


def mann_whitney_feature_selection(df, label_col="Label", p_thresh=0.1):

    selected_features = []
    p_values = {}

    y = df[label_col]
    feature_cols = [c for c in df.columns if c != label_col]

    group0 = df[y == 0]
    group1 = df[y == 1]

    for col in feature_cols:
        x0 = group0[col].dropna()
        x1 = group1[col].dropna()

        if len(x0) == 0 or len(x1) == 0:
            continue

        try:
            _, p = mannwhitneyu(x0, x1, alternative="two-sided")
        except ValueError:
            p = 1.0

        p_values[col] = p
        if p < p_thresh:
            selected_features.append(col)

    p_df = pd.DataFrame({
        "feature": list(p_values.keys()),
        "p_value": list(p_values.values())
    }).sort_values("p_value", ascending=True)

    return selected_features, p_df


def find_optimal_cutoff_by_youden(y_true, y_prob):
    """
    Youden index = sensitivity + specificity - 1 = TPR - FPR
    """
    fpr, tpr, thresholds = roc_curve(y_true, y_prob)
    youden_index = tpr - fpr
    best_idx = np.argmax(youden_index)
    best_threshold = thresholds[best_idx]
    return best_threshold, fpr, tpr, thresholds, youden_index


def compute_metrics(y_true, y_prob, threshold):
 
    y_pred = (y_prob >= threshold).astype(int)

    auc = roc_auc_score(y_true, y_prob)
    acc = accuracy_score(y_true, y_pred)
    f1 = f1_score(y_true, y_pred)
    rec = recall_score(y_true, y_pred)
    prec = precision_score(y_true, y_pred)

    cm = confusion_matrix(y_true, y_pred, labels=[0, 1])
    # cm = [[TN, FP],
    #       [FN, TP]]
    tn, fp, fn, tp = cm.ravel()

    sensitivity = tp / (tp + fn) if (tp + fn) > 0 else np.nan
    specificity = tn / (tn + fp) if (tn + fp) > 0 else np.nan
    ppv = tp / (tp + fp) if (tp + fp) > 0 else np.nan
    npv = tn / (tn + fn) if (tn + fn) > 0 else np.nan

    return {
        "AUC": auc,
        "Accuracy": acc,
        "Sensitivity": sensitivity,
        "Specificity": specificity,
        "F1": f1,
        "Recall": rec,
        "Precision": prec,
        "PPV": ppv,
        "NPV": npv
    }


if __name__ == "__main__":
   
    train_path = r"./dfKfeas_train.csv"
    test_path = r"./dfKfeas_test.csv"

    label_col = "Label"
    id_col = "ID"
    random_state = 42

   
    train_df = pd.read_csv(train_path)
    test_df = pd.read_csv(test_path)

    train_ids = train_df[id_col].copy() if id_col in train_df.columns else None
    test_ids = test_df[id_col].copy() if id_col in test_df.columns else None

    if id_col in train_df.columns:
        train_df = train_df.drop(columns=[id_col])
    if id_col in test_df.columns:
        test_df = test_df.drop(columns=[id_col])

    common_cols = [c for c in train_df.columns if c in test_df.columns]
    train_df = train_df[common_cols]
    test_df = test_df[common_cols]

    train_df = train_df.dropna(subset=[label_col]).reset_index(drop=True)
    test_df = test_df.dropna(subset=[label_col]).reset_index(drop=True)


    feature_cols_all = [c for c in train_df.columns if c != label_col]

    imputer = SimpleImputer(strategy="median")
    train_df[feature_cols_all] = imputer.fit_transform(train_df[feature_cols_all])
    test_df[feature_cols_all] = imputer.transform(test_df[feature_cols_all])

    mw_features, mw_pvalues = mann_whitney_feature_selection(
        train_df, label_col=label_col, p_thresh=0.1
    )

    if len(mw_features) == 0:
        raise ValueError("曼-惠特尼 U 检验后没有特征满足 P < 0.1，请检查数据。")

    print(f"曼-惠特尼 U 检验后保留特征数: {len(mw_features)}")

    mw_pvalues.to_csv("mannwhitney_pvalues.csv", index=False, encoding="utf-8-sig")


    X_train_mw = train_df[mw_features].copy()
    y_train = train_df[label_col].values.astype(int)

    X_test_mw = test_df[mw_features].copy()
    y_test = test_df[label_col].values.astype(int)


    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train_mw)
    X_test_scaled = scaler.transform(X_test_mw)

    svm_estimator = SVC(kernel="linear", random_state=random_state)

    rfecv = RFECV(
        estimator=svm_estimator,
        step=1,
        cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=random_state),
        scoring="roc_auc",
        n_jobs=-1,
        min_features_to_select=1
    )

    rfecv.fit(X_train_scaled, y_train)

    selected_mask = rfecv.support_
    selected_features = list(np.array(mw_features)[selected_mask])

    print(f"SVM-RFE 最终保留特征数: {len(selected_features)}")
    print("最终特征:", selected_features)

    X_train_final = train_df[selected_features].copy()
    X_test_final = test_df[selected_features].copy()


    rf = RandomForestClassifier(random_state=random_state)

    param_grid = {
        "n_estimators": [100, 200, 300, 500],
        "max_depth": [None, 3, 4, 5, 6],
        "min_samples_split": [2, 3, 4, 5],
        "min_samples_leaf": [1, 2, 3],
        "max_features": ["sqrt", "log2", None],
        "bootstrap": [True, False],
        "criterion": ["gini", "entropy"],
        "class_weight": [None, "balanced"]
    }

    cv_strategy = StratifiedKFold(n_splits=5, shuffle=True, random_state=random_state)

    grid_search = GridSearchCV(
        estimator=rf,
        param_grid=param_grid,
        scoring="roc_auc",
        cv=cv_strategy,
        n_jobs=-1,
        verbose=1
    )

    grid_search.fit(X_train_final, y_train)

    best_model = grid_search.best_estimator_
    print("最佳参数:", grid_search.best_params_)
    print("训练集5折CV最佳AUC:", grid_search.best_score_)


    train_prob = best_model.predict_proba(X_train_final)[:, 1]
    test_prob = best_model.predict_proba(X_test_final)[:, 1]

    best_threshold, fpr, tpr, thresholds, youden_index = find_optimal_cutoff_by_youden(
        y_train, train_prob
    )

    print(f"\n基于训练集最大约登指数的最佳阈值: {best_threshold:.6f}")


    train_metrics = compute_metrics(y_train, train_prob, threshold=best_threshold)
    test_metrics = compute_metrics(y_test, test_prob, threshold=best_threshold)

    print("\n===== 训练集指标 =====")
    for k, v in train_metrics.items():
        print(f"{k}: {v:.4f}")

    print("\n===== 测试集指标 =====")
    for k, v in test_metrics.items():
        print(f"{k}: {v:.4f}")


    train_pred_df = pd.DataFrame({
        "ID": train_ids if train_ids is not None else np.arange(len(train_df)),
        "Label": y_train,
        "PR_prob": train_prob,
        "PR_pred": (train_prob >= best_threshold).astype(int)
    })

    test_pred_df = pd.DataFrame({
        "ID": test_ids if test_ids is not None else np.arange(len(test_df)),
        "Label": y_test,
        "PR_prob": test_prob,
        "PR_pred": (test_prob >= best_threshold).astype(int)
    })

    train_pred_df.to_csv("train_PR_probability.csv", index=False)
    test_pred_df.to_csv("test_PR_probability.csv", index=False)


    selected_feature_df = pd.DataFrame({
        "selected_features": selected_features
    })
    selected_feature_df.to_csv("selected_features_rfecv.csv", index=False, encoding="utf-8-sig")

    youden_df = pd.DataFrame({
        "threshold": thresholds,
        "TPR_sensitivity": tpr,
        "FPR": fpr,
        "youden_index": youden_index
    }).sort_values("youden_index", ascending=False)

    youden_df.to_csv("youden_index_table.csv", index=False, encoding="utf-8-sig")

    print("\n结果已保存：")
    print("- mannwhitney_pvalues.csv")
    print("- selected_features_rfecv.csv")
    print("- train_PR_probability.csv")
    print("- test_PR_probability.csv")
    print("- youden_index_table.csv")